# ADRec Baby: without warm-up

Compare new ADRec runs with `warmup_epochs = 0` against the saved five-epoch warm-up runs, using seeds 42–46. Each new model starts from its seed's original pretrained item embeddings. No embedding pretraining is repeated, and no finished ADRec model is used to initialize training.

Full runs retain 32 diffusion steps and every other saved baseline configuration setting. A one-epoch smoke check runs first, using nine users and the original embedding dimension. Smoke outputs are separate from full results.

1. Upload `adrec_baby_warmup0_inputs.zip` as a private Kaggle dataset and attach it to this notebook. It contains the code, Baby dataset, five pretrained checkpoints, and baseline metadata.
2. Enable Internet for dependency installation and select a GPU.
3. Run all cells. `RUN_FULL = True` performs the smoke check followed by all five full ADRec runs. Set it to `False` for the smoke check only.

Outputs are saved in `/kaggle/working`; ZIP links appear at the end. A browser download is a separate step.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
INPUT_BUNDLE = None  # Optional exact path to the input ZIP or its extracted folder.
RESUME_ROOT = None  # Optional folder containing adrec_warmup0_seed42/, ..., seed46/.
RUN_FULL = True
SEEDS = [42, 43, 44, 45, 46]
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## 1. Load and verify the input bundle

The setup accepts either the intact upload ZIP or Kaggle-extracted files. SHA-256 checks verify the dataset, baseline records, pretrained weights, and source files. If several copies are attached, set `INPUT_BUNDLE` to one exact ZIP or extracted folder. Start a fresh kernel when changing the source bundle.


In [ ]:
import hashlib
import json
import shutil
import sys
import zipfile


def read_json(path):
    return json.loads(Path(path).read_text())


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def inside(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if path == root or root not in path.parents:
        raise ValueError(f'Unsafe archive or manifest path: {name}')
    return path


def unpack(archive, destination):
    destination = Path(destination)
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            inside(destination, item.filename)
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Archive links are not supported')
        bundle.extractall(destination)
    return destination


def locate_inputs(root, override, cache):
    root = Path(override) if override else Path(root)
    if root.is_file():
        root = unpack(root, cache)
    manifests = sorted(root.rglob('warmup_inputs_manifest.json'))
    if not manifests and override is None:
        archives = sorted(root.rglob('adrec_baby_warmup0_inputs.zip'))
        if len(archives) != 1:
            raise ValueError('Attach one warm-up input bundle or set INPUT_BUNDLE.')
        root = unpack(archives[0], cache)
        manifests = sorted(root.rglob('warmup_inputs_manifest.json'))
    if len(manifests) != 1:
        raise ValueError(f'Expected one warm-up manifest; found {manifests}')
    manifest = read_json(manifests[0])
    if manifest['experiment'] != 'adrec_baby_warmup0' or manifest['seeds'] != [42, 43, 44, 45, 46]:
        raise ValueError('Wrong experiment bundle')
    root = manifests[0].parent
    for name, expected in manifest['files'].items():
        if sha256(inside(root, name)) != expected:
            raise ValueError(f'Input hash mismatch: {name}')
    return root, manifest


def prepare_code(inputs, repo, manifest):
    inputs, repo = Path(inputs), Path(repo)
    if repo.exists():
        raise FileExistsError('Source directory already exists. Restart with a fresh WORK_ROOT or reuse the verified REPO variable.')
    manifests = sorted(inputs.rglob('bundle_manifest.json'))
    if len(manifests) == 1:
        shutil.copytree(manifests[0].parent, repo)
    elif not manifests and (inputs / 'cmpe257_kaggle_code.zip').is_file():
        unpack(inputs / 'cmpe257_kaggle_code.zip', repo)
    else:
        raise ValueError('Expected one source ZIP or one extracted source manifest')
    if sha256(repo / 'bundle_manifest.json') != manifest['code_manifest_sha256']:
        raise ValueError('Wrong code bundle')
    source = read_json(repo / 'bundle_manifest.json')
    for name, expected in source['files'].items():
        if sha256(inside(repo, name)) != expected:
            raise ValueError(f'Source hash mismatch: {name}')
    return repo


In [ ]:
INPUTS, INPUT_MANIFEST = locate_inputs(INPUT_ROOT, INPUT_BUNDLE, WORK_ROOT / 'warmup_input_cache')
REPO_PATH = WORK_ROOT / 'CMPE257_ML_warmup0'
if not REPO_PATH.exists():
    REPO = prepare_code(INPUTS, REPO_PATH, INPUT_MANIFEST)
else:
    REPO = REPO_PATH
    if sha256(REPO / 'bundle_manifest.json') != INPUT_MANIFEST['code_manifest_sha256']:
        raise ValueError('Existing code is from a different bundle; start a fresh session.')
    for name, expected in read_json(REPO / 'bundle_manifest.json')['files'].items():
        if sha256(inside(REPO, name)) != expected:
            raise ValueError(f'Existing source changed: {name}')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
DATASET_PATH = INPUTS / 'dataset.pkl'
BASELINE = INPUTS / 'baseline'
print('Code:', REPO)
print('Data:', DATASET_PATH)
print('Pretrained seeds:', INPUT_MANIFEST['seeds'])


## 2. Install dependencies and check the runtime

Install the shared requirements. If installation requests a kernel restart, restart and run the notebook from the beginning. Full training uses one GPU.


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-shared.txt')])


In [ ]:
import torch
from src.training.kaggle import train, make_config
from src.data import load_dataset
from src.data.load import dataset_summary

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
if RUN_FULL and not torch.cuda.is_available():
    raise RuntimeError('Select a GPU for full runs, or set RUN_FULL = False for a CPU smoke check.')
print(dataset_summary(load_dataset(DATASET_PATH, INPUT_MANIFEST['dataset_sha256'])))


## 3. Check baseline compatibility and define the experiment

The baseline configurations are the source of full-run settings. Only `warmup_epochs` changes from 5 to 0. The saved pretraining checkpoint must match both the seed and the checkpoint hash recorded by the original run.

Source checks allow only the previously audited Baby dataset-support migration. Other changes to training, model, or evaluation files stop the experiment. Runtime versions are recorded separately; different Kaggle environments can still affect numerical results.

Completed runs are reused; interrupted runs resume from `latest.pt`. Resume requires identical configuration, pretrained weights, and source files.


In [ ]:
import csv
import math
import statistics

METRICS = ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']
METADATA = ['config.json', 'results.json', 'history.json', 'environment.json', 'requirements-lock.txt', 'experiment_input.json']
RUNS_ROOT = WORK_ROOT / 'runs/adrec_warmup_ablation/baby'
RESULTS_DIR = WORK_ROOT / 'results/adrec_warmup_ablation/baby'
ARCHIVE_ROOT = WORK_ROOT / 'exports'
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)


def variant_config(base, smoke=False):
    cfg = dict(base, warmup_epochs=0)
    if smoke:
        # Keep the trained embedding shape, max_len, and 32 diffusion steps.
        cfg.update(mode='smoke', user_limit=9, train_batch_size=4,
                   evaluation_batch_size=4, epochs=1, eval_interval=1)
    return cfg


def check_report(cfg, report):
    if report['model'] != 'adrec' or report['mode'] != cfg['mode']:
        raise ValueError('Result model/mode mismatch')
    test = report.get('test')
    if not test or test['protocol'] != 'adrec_upstream' or test['units'] != 'percent':
        raise ValueError('Expected completed upstream-protocol evaluation')
    expected_users = cfg['user_limit'] or 11761
    if report['users'] != expected_users or test['users'] != expected_users:
        raise ValueError('Unexpected evaluation population')
    for key in METRICS:
        value = test['metrics'][key]
        if not math.isfinite(value) or not 0 <= value <= 100:
            raise ValueError(f'Invalid metric: {key}')


def validate_references():
    references, migrations = {}, []
    allowed = read_json(REPO / 'configs/legacy_source_compatibility.json')['migrations']
    current = read_json(REPO / 'bundle_manifest.json')['files']
    for seed in SEEDS:
        cfg = read_json(BASELINE / f'adrec_seed{seed}_config.json')
        expected = make_config('adrec', 'full')
        expected['seed'] = seed
        if cfg != expected or cfg['warmup_epochs'] != 5 or cfg['diffusion_steps'] != 32:
            raise ValueError(f'Baseline configuration mismatch for seed {seed}')
        report = read_json(BASELINE / f'adrec_seed{seed}_results.json')
        check_report(cfg, report)
        env = read_json(BASELINE / f'adrec_seed{seed}_environment.json')
        for name, old in env['source']['files'].items():
            relevant = ((name.startswith(('src/', 'third_party/ADRec/src/')) and name.endswith(('.py', '.yaml')))
                        or name in ('configs/shared.json', 'configs/kaggle.json'))
            if not relevant:
                continue
            new = current.get(name)
            if old != new:
                matches = [m for m in allowed if m['dataset'] == 'baby' and m['path'] == name
                           and m['before'] == old and m['after'] == new]
                if len(matches) != 1:
                    raise ValueError(f'Unreviewed source difference: {name}')
                migrations.append({'seed': seed, **matches[0]})
        pretrain = INPUTS / f'pretrain/seed{seed}/best.pt'
        recorded = read_json(BASELINE / f'adrec_seed{seed}_checkpoints.json')['files']['pretrain/best.pt']
        if sha256(pretrain) != recorded or env['pretrained_checkpoint']['sha256'] != recorded:
            raise ValueError(f'Pretraining checkpoint mismatch: seed {seed}')
        saved = torch.load(pretrain, map_location='cpu', weights_only=True)
        pre_cfg = read_json(BASELINE / f'pretrain/pretrain_seed{seed}_config.json')
        if saved['config'] != pre_cfg or pre_cfg != dict(cfg, model='pretrain'):
            raise ValueError(f'Pretraining config/seed mismatch: seed {seed}')
        if saved['num_items'] != 4731 or tuple(saved['model_state']['item_embedding.weight'].shape) != (4732, cfg['hidden_size']):
            raise ValueError('Wrong embedding vocabulary or dimension')
        references[seed] = {'config': cfg, 'report': report, 'pretrain': pretrain}
    return references, migrations


def ensure_run(cfg, pretrain, output, prior=None):
    output = Path(output)
    identity = {'config': cfg, 'pretrained_sha256': sha256(pretrain),
                'code_manifest_sha256': INPUT_MANIFEST['code_manifest_sha256']}
    if not (output / 'latest.pt').exists() and prior and Path(prior).is_dir():
        if output.exists() and any(output.iterdir()):
            raise ValueError(f'Resume destination is not empty: {output}')
        shutil.copytree(prior, output, dirs_exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    marker = output / 'experiment_input.json'
    if marker.exists() and read_json(marker) != identity:
        raise ValueError(f'Run input mismatch: {output}')
    resume = output / 'latest.pt'
    if resume.exists() and not marker.exists():
        raise ValueError('Saved run is missing input provenance')
    write_json(marker, identity)
    if resume.exists():
        if read_json(output / 'config.json') != cfg:
            raise ValueError('Resume configuration mismatch')
        report_path = output / 'results.json'
        report = read_json(report_path) if report_path.exists() else None
        if report and (report['early_stopped'] or report['epochs_completed'] >= cfg['epochs']) and report.get('test') and (output / 'best.pt').is_file():
            check_report(cfg, report)
            print('Reusing completed run:', output.name)
            return report
    return train(cfg, DATASET_PATH, output, pretrained_checkpoint=pretrain,
                 resume=resume if resume.exists() else None, device=DEVICE, finalize=True)


def write_csv(path, columns, rows):
    with Path(path).open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def summarize_rows(rows):
    summary = []
    for variant in sorted({row['variant'] for row in rows}):
        group = sorted([r for r in rows if r['variant'] == variant], key=lambda r: r['seed'])
        common = {'variant': variant, 'model': 'ADRec', 'dataset': 'Amazon Baby',
                  'warmup_epochs': group[0]['warmup_epochs'], 'n_runs': len(group),
                  'seeds': ','.join(str(r['seed']) for r in group)}
        summary.append({**common, 'statistic': 'mean',
                        **{m: round(statistics.mean(r[m] for r in group), 4) for m in METRICS}})
        if len(group) > 1:
            summary.append({**common, 'statistic': 'std (sample, n-1)',
                            **{m: round(statistics.stdev(r[m] for r in group), 4) for m in METRICS}})
    return summary


def export_results(references, migrations):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    rows, paired = [], []
    for seed, ref in references.items():
        output = RUNS_ROOT / f'adrec_warmup0_seed{seed}'
        if not (output / 'results.json').is_file():
            continue
        cfg = read_json(output / 'config.json')
        if cfg != variant_config(ref['config']):
            raise ValueError('Only matching full runs belong in this comparison')
        result = read_json(output / 'results.json')
        check_report(cfg, result)
        if not (result['early_stopped'] or result['epochs_completed'] >= cfg['epochs']):
            continue
        for name in METADATA:
            shutil.copyfile(output / name, RESULTS_DIR / f'adrec_seed{seed}_{name}')
        write_json(RESULTS_DIR / f'adrec_seed{seed}_checkpoints.json', {
            'seed': seed, 'pretrained_sha256': sha256(ref['pretrain']),
            'location': f'checkpoints/adrec_warmup_ablation/baby/{output.name}',
            'files': {name: sha256(output / name) for name in ['best.pt', 'latest.pt']}})
        for variant, warmup, report in [('warmup5', 5, ref['report']), ('warmup0', 0, result)]:
            rows.append({'variant': variant, 'run_id': f'adrec_{variant}_seed{seed}', 'model': 'ADRec',
                         'dataset': 'Amazon Baby', 'seed': seed, 'warmup_epochs': warmup,
                         'best_epoch': report['best_epoch'], 'epochs_completed': report['epochs_completed'],
                         **report['test']['metrics']})
        paired.append({'seed': seed, **{m: round(result['test']['metrics'][m] - ref['report']['test']['metrics'][m], 4) for m in METRICS}})
    if not rows:
        return []
    write_csv(RESULTS_DIR / 'accuracy_runs.csv', ['variant','run_id','model','dataset','seed','warmup_epochs','best_epoch','epochs_completed',*METRICS], rows)
    write_csv(RESULTS_DIR / 'accuracy_summary.csv', ['variant','model','dataset','warmup_epochs','n_runs','seeds','statistic',*METRICS], summarize_rows(rows))
    write_csv(RESULTS_DIR / 'paired_differences.csv', ['seed',*METRICS], paired)
    # The baseline metadata remains unmodified and is included for provenance.
    shutil.copytree(BASELINE, RESULTS_DIR / 'baseline', dirs_exist_ok=True)
    write_json(RESULTS_DIR / 'source_migrations.json', migrations)
    write_json(RESULTS_DIR / 'input_manifest.json', INPUT_MANIFEST)
    write_json(RESULTS_DIR / 'completion.json', {'expected_seeds': SEEDS,
               'completed_seeds': [r['seed'] for r in paired], 'complete': len(paired) == len(SEEDS)})
    write_csv(RESULTS_DIR / 'paper_reference.csv', ['variant','HR@20','NDCG@20','source'], [
        {'variant':'warmup5','HR@20':7.1524,'NDCG@20':3.1455,'source':'https://arxiv.org/html/2505.19544v1#S4.SS4'},
        {'variant':'warmup0','HR@20':6.7507,'NDCG@20':2.9535,'source':'https://arxiv.org/html/2505.19544v1#S4.SS4'}])
    (RESULTS_DIR / 'EXPERIMENT.md').write_text("# ADRec Baby warm-up ablation\n\nNew runs set warmup_epochs=0; historical baseline runs used five epochs. Both use 32 diffusion steps and the same per-seed pretrained embeddings. Only warmup_epochs changes in the full-run configuration. New ADRec backbones start fresh; pretraining is not repeated.\n\naccuracy_runs.csv and accuracy_summary.csv compare the same completed seeds in both conditions. Summaries report mean and sample standard deviation. Partial batches are labeled in completion.json; missing seeds are not filled. paired_differences.csv reports warmup0 minus warmup5 in percentage points.\n\nBaseline results are reused as recorded, not re-evaluated or retrained. The exact audited Baby source migration is recorded in source_migrations.json. Runtime and package versions may differ between historical and new runs. HR/NDCG use the existing upstream-style evaluator. Checkpoints are selected by validation with the same early-stopping settings. Our baseline strictly freezes embeddings during warm-up; this differs from the released optimizer's possible weight-decay updates. These limitations remain applicable to comparisons with the paper.\n\npaper_reference.csv contains published Baby point estimates from Table 4, https://arxiv.org/html/2505.19544v1#S4.SS4. They are separate from project measurements and have no invented per-seed values or standard deviations. No inference-speed benchmark is part of this experiment.\n")
    return rows


def save_archives():
    paths = []
    if RESULTS_DIR.exists():
        paths.append(shutil.make_archive(str(ARCHIVE_ROOT / 'adrec_baby_warmup0_results'), 'zip', root_dir=RESULTS_DIR))
    if RUNS_ROOT.exists():
        paths.append(shutil.make_archive(str(ARCHIVE_ROOT / 'adrec_baby_warmup0_models'), 'zip', root_dir=RUNS_ROOT))
    return paths


In [ ]:
REFERENCES, SOURCE_MIGRATIONS = validate_references()
print('Validated all five original pretrained checkpoints and baseline configurations.')
for seed, ref in REFERENCES.items():
    cfg = variant_config(ref['config'])
    changed = {k: (ref['config'][k], cfg[k]) for k in cfg if cfg[k] != ref['config'][k]}
    print('Seed', seed, 'changes:', changed)


## 4. Smoke check: embeddings update in epoch zero

Use the saved seed-42 pretrained embeddings, the full 128-dimensional model, and 32 diffusion steps on nine users for one epoch. This checks actual checkpoint loading, training, evaluation, and serialization. The test verifies that embeddings are trainable and change during the first epoch. Its scores are not included in the full comparison.


In [ ]:
ref = REFERENCES[42]
smoke_cfg = variant_config(ref['config'], smoke=True)
smoke_dir = WORK_ROOT / 'smoke_runs/adrec_warmup_ablation/baby/seed42'
smoke_report = ensure_run(smoke_cfg, ref['pretrain'], smoke_dir)
initial = torch.load(ref['pretrain'], map_location='cpu', weights_only=True)['model_state']['item_embedding.weight']
latest = torch.load(smoke_dir / 'latest.pt', map_location='cpu', weights_only=True)
if latest['history'][0]['epoch'] != 0 or not latest['history'][0]['embedding_trainable']:
    raise AssertionError('Embeddings must be trainable from epoch zero')
change = torch.linalg.vector_norm(latest['model_state']['item_embedding.weight'] - initial).item()
if not math.isfinite(change) or change <= 0:
    raise AssertionError('Expected updated embeddings after epoch zero')
SMOKE_CHECK = {'passed': True, 'seed': 42, 'epoch': 0, 'embedding_change_l2': change,
               'config': smoke_cfg, 'pretrained_sha256': sha256(ref['pretrain']),
               'code_manifest_sha256': INPUT_MANIFEST['code_manifest_sha256']}
write_json(smoke_dir / 'smoke_check.json', SMOKE_CHECK)
del initial, latest
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Smoke check passed. Embedding change L2:', change)


## 5. Train and export seeds 42–46

For each seed, load its pretrained embeddings into a newly initialized ADRec backbone and train all parameters from epoch zero. The original validation selection, early stopping, and final test evaluation remain unchanged.

Result and model ZIPs are refreshed after every seed. Rerunning this cell reuses completed runs and resumes unfinished runs. For another session, attach the extracted model ZIP and set `RESUME_ROOT` to the folder containing `adrec_warmup0_seed42/` through `adrec_warmup0_seed46/`. Keep the original input bundle attached.


In [ ]:
if RUN_FULL:
    if not SMOKE_CHECK['passed']:
        raise RuntimeError('Complete the smoke check before full training.')
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    write_json(RESULTS_DIR / 'smoke_check.json', SMOKE_CHECK)
    for seed in SEEDS:
        ref = REFERENCES[seed]
        output = RUNS_ROOT / f'adrec_warmup0_seed{seed}'
        prior = Path(RESUME_ROOT) / output.name if RESUME_ROOT else None
        try:
            report = ensure_run(variant_config(ref['config']), ref['pretrain'], output, prior)
            export_results(REFERENCES, SOURCE_MIGRATIONS)
            print('Completed seed:', seed, report['test']['metrics'])
        finally:
            # Preserve completed results and any epoch checkpoint if a run stops.
            for path in save_archives():
                print('Saved:', path)
else:
    print('Smoke-only execution complete. Set RUN_FULL = True to train all five seeds.')


## 6. Inspect and download outputs

`accuracy_summary.csv` reports the mean and sample standard deviation for the same completed seeds in both conditions. `paired_differences.csv` reports zero warm-up minus five warm-up in percentage points. `paper_reference.csv` contains separate published values, not new measurements.

Download both ZIPs from `/kaggle/working/exports/`:

- `adrec_baby_warmup0_results.zip` → extract into `results/adrec_warmup_ablation/baby/` locally. This contains metadata and CSVs, without weights.
- `adrec_baby_warmup0_models.zip` → extract into `checkpoints/adrec_warmup_ablation/baby/` locally. This contains `best.pt`, `latest.pt`, and run metadata for each completed or interrupted seed.

The full input bundle and original baseline files are preserved. Keep model weights outside Git. Kaggle **Save Version → Save & Run All** starts a fresh background execution; attach saved outputs through `RESUME_ROOT` to continue earlier work. The notebook creates archives but does not automatically download them to your laptop.


In [ ]:
try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = str
    display = print

summary_path = RESULTS_DIR / 'accuracy_summary.csv'
if summary_path.exists():
    print(summary_path.read_text())
    print('Completion:', read_json(RESULTS_DIR / 'completion.json'))
    print('Paper reference (percent):')
    print((RESULTS_DIR / 'paper_reference.csv').read_text())
for path in save_archives():
    display(FileLink(path))
